Analysation of the data coming from the simulator.

In [ ]:
import yaml
import math
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from pathlib import Path, PurePath

In [ ]:
environment = "arena_perpendicular"

dataset_path = Path("wireless_channel/sionna_dataset/", environment)

# Read out the config file.
config_file = f"environments/{environment}/config.yaml"
with open(config_file, "r", encoding="utf8") as file:
    config = yaml.safe_load(file)

stripe_config = config["stripe_config"]

In [ ]:
def find_closest_ru(data, ue, n_stripes, n_rus):
    """
    data: nested list of dictionaries
    ue: dict with 'x', 'y', 'z' of the UE
    """

    ue_x, ue_y, ue_z = ue["x"], ue["y"], ue["z"]

    closest_distance = float("inf")
    closest_index = None
    closest_ru_coords = None

    # Loop over outer list (groups)
    ru_index = 0
    for group in data:

        # Loop over elements inside each group
        for entry in group:
            if "radio_unit" in entry:  # skip central_unit
                ru = entry["radio_unit"]
                dx = ru["x"] - ue_x
                dy = ru["y"] - ue_y
                dz = ru["z"] - ue_z
                dist = math.sqrt(dx**2 + dy**2 + dz**2)

                if dist < closest_distance:
                    closest_distance = dist
                    closest_index = ru_index
                    closest_ru_coords = (ru["x"], ru["y"], ru["z"])

                ru_index += 1
            
    closest_stripe = closest_index // n_rus
    closest_ru = closest_index % n_rus

    return closest_distance, closest_ru_coords, closest_stripe, closest_ru

def calculate_distance(ue, ru_coords, dstripe, dru, start):
    """Calculate the distance between the UE and the RU."""

    ue_x, ue_y = ue["x"], ue["y"]
    ru_x, ru_y = ru_coords[0], ru_coords[1]
    startx, starty = start

    dx = (ru_x * dstripe) + startx - ue_x
    dy = (ru_y * dru) + starty - ue_y
    dist = math.sqrt(dx*dx + dy*dy)

    return dist

In [ ]:
find_closest_ru(config['radio_stripes'], {'x': 9.0, 'y': 21.5, 'z': 1.0}, 8, 20)

In [ ]:
file_path = PurePath(dataset_path, "ue_locations/ue_locations.nc")
ue_ds = xr.load_dataset(file_path)

nmse_optimized = []
dist_optimized = []
for ue_id in range(len(ue_ds['user_id'])):
    user = ue_ds.where(ue_ds["user_id"] == ue_id, drop=True)
    ue_x = float(user["x"])
    ue_y = float(user["y"])
    ue_z = float(user["z"])

    n_stripes = stripe_config['N_stripes']
    n_rus = stripe_config['N_RUs']
    dstripe = stripe_config['space_between_stripes']
    dru = stripe_config['space_between_RUs']
    startx, starty, startz = stripe_config['stripe_start_pos']
    dist, ccoords, stripe_idx, ru_idx = find_closest_ru(config['radio_stripes'], user, n_stripes, n_rus)

    # Read in the data for UEx
    try:
        file_path = PurePath(dataset_path, f"flickering/flickering_data_{ue_id}.pkl")
        df = pd.read_pickle(file_path)
    except FileNotFoundError:
        continue

    # For this UE select the RU with the best NMSE and calculate the distance to that RU.
    row = df[df["nmse"] == df["nmse"].min()]
    # For this UE select the closest RU.
    row2 = df[(df["stripe_id"] == stripe_idx) & (df["ru_id"] == ru_idx)]
    row2 = row2[row2["nmse"] == row2["nmse"].min()]
    nmse_optimized.append(row)
    dist_optimized.append(row2)

nmse_df = pd.concat(nmse_optimized)
dist_df = pd.concat(dist_optimized)

# Now calculate the CDF for both scenarios and plot it.
grid_start = 500
sndr_sorted_rand = np.sort(-nmse_df['nmse'][:grid_start])
sndr_cdf_rand = np.arange(1, len(sndr_sorted_rand) + 1) / len(sndr_sorted_rand)
sndr_sorted_struct = np.sort(-nmse_df['nmse'][grid_start:])
sndr_cdf_struct = np.arange(1, len(sndr_sorted_struct) + 1) / len(sndr_sorted_struct)
sndr_sorted_full = np.sort(-nmse_df['nmse'])
sndr_cdf_full = np.arange(1, len(sndr_sorted_full) + 1) / len(sndr_sorted_full)

# Calculate CDF for distance
dist_sorted_rand = np.sort(-dist_df['nmse'][:grid_start])
dist_cdf_rand = np.arange(1, len(dist_sorted_rand) + 1) / len(dist_sorted_rand)
dist_sorted_struct = np.sort(-dist_df['nmse'][grid_start:])
dist_cdf_struct = np.arange(1, len(dist_sorted_struct) + 1) / len(dist_sorted_struct)
dist_sorted_full = np.sort(-dist_df['nmse'])
dist_cdf_full = np.arange(1, len(dist_sorted_full) + 1) / len(dist_sorted_full)

# Plot the CDFs
fig, ax = plt.subplots()
ax.plot(sndr_sorted_full, sndr_cdf_full, color='C0', label='SNDR all UEs')
ax.plot(dist_sorted_full, dist_cdf_full, color='C1', label='Distance all UEs')
ax.plot(sndr_sorted_rand, sndr_cdf_rand, '--', color='C0', label='SNDR random UEs')
ax.plot(dist_sorted_rand, dist_cdf_rand, '--', color='C1', label='Distance random UEs')
ax.plot(sndr_sorted_struct, sndr_cdf_struct, '-.', color='C0', label='SNDR grid UEs')
ax.plot(dist_sorted_struct, dist_cdf_struct, '-.', color='C1', label='Distance grid UEs')
ax.set_xlabel('SNDR [dB]')
ax.set_ylabel('CDF')
ax.legend()

df = pd.DataFrame(np.array([sndr_sorted_full, sndr_cdf_full, dist_sorted_full, dist_cdf_full]).T, columns=["SNDR SNDR", "SNDR CDF", "Dist SNDR", "Dist CDF"])
file_path = PurePath(dataset_path, "flickering/cdf_sndr_all.csv")
df.to_csv(file_path, index=False)
df = pd.DataFrame(np.array([sndr_sorted_rand, sndr_cdf_rand, dist_sorted_rand, dist_cdf_rand]).T, columns=["SNDR SNDR", "SNDR CDF", "Dist SNDR", "Dist CDF"])
file_path = PurePath(dataset_path, "flickering/cdf_sndr_rand.csv")
df.to_csv(file_path, index=False)
df = pd.DataFrame(np.array([sndr_sorted_struct, sndr_cdf_struct, dist_sorted_struct, dist_cdf_struct]).T, columns=["SNDR SNDR", "SNDR CDF", "Dist SNDR", "Dist CDF"])
file_path = PurePath(dataset_path, "flickering/cdf_sndr_struct.csv")
df.to_csv(file_path, index=False)

file_path = PurePath(dataset_path, "flickering/cdf_sndr.pdf")
fig.savefig(file_path)